# 00 — H&M data preparation

Standalone H&M Kaggle pipeline.

## Discover the mounted Kaggle dataset and lock configuration

In [ ]:
# Kaggle: attach the competition data as an Input, then run this cell.
from pathlib import Path
import os, json

def find_file(name: str) -> Path:
    roots = [Path("/kaggle/input"), Path.cwd()]
    for root in roots:
        if not root.exists(): continue
        for p in root.glob(f"*/{name}"): return p
        for p in root.glob(name): return p
        match = next((p for p in root.rglob(name) if "images" not in p.parts), None)
        if match: return match
    raise FileNotFoundError(f"Cannot find {name}. Attach the H&M competition dataset as a Kaggle Input.")

files = {name: find_file(name) for name in ("transactions_train.csv", "articles.csv", "customers.csv")}
dataset_dir = files["articles.csv"].parent
image_dirs = [p for p in [dataset_dir / "images", Path("/kaggle/input/h-and-m-personalized-fashion-recommendations/images")] if p.exists()]
if not image_dirs:
    image_dirs = [p for p in Path("/kaggle/input").glob("**/images") if p.is_dir()]
profile = os.getenv("HM_PROFILE", "quick").lower()
assert profile in {"quick", "full"}
work = Path(os.getenv("HM_WORK_DIR", "/kaggle/working/hm_v1")); work.mkdir(parents=True, exist_ok=True)
config = {
    "dataset": "H&M Personalized Fashion Recommendations", "version": "hm_v1", "profile": profile,
    "seed": 20260922, "customers": 50_000 if profile == "quick" else 300_000,
    "min_train_events": 3, "max_events_per_customer": 100,
    "transactions": str(files["transactions_train.csv"]), "articles": str(files["articles.csv"]),
    "customers_file": str(files["customers.csv"]), "images": str(image_dirs[0]) if image_dirs else None,
}
(work / "run_config.json").write_text(json.dumps(config, indent=2))
for name, path in files.items(): print(f"{name}: {path} ({path.stat().st_size / 1e9:.2f} GB)")
print("images:", config["images"], "\nworking directory:", work)

Do not use `kaggle competitions download` inside this notebook. Mounting the competition data is more reliable, avoids credentials, and works with Internet disabled.

## Smart sample, chronological windows, and Parquet ETL

In [ ]:
from pathlib import Path
import os, json, random, hashlib
import numpy as np
import polars as pl

PROFILE = os.getenv("HM_PROFILE", "quick").lower()
assert PROFILE in {"quick", "full"}
WORK = Path(os.getenv("HM_WORK_DIR", "/kaggle/working/hm_v1"))
WORK.mkdir(parents=True, exist_ok=True)
cfg = json.loads((WORK / "run_config.json").read_text())
SEED = cfg["seed"]
random.seed(SEED); np.random.seed(SEED)


In [ ]:
from datetime import timedelta
tx = cfg["transactions"]; articles_csv = cfg["articles"]
schema = pl.scan_csv(tx, schema_overrides={"article_id": pl.String, "customer_id": pl.String}, try_parse_dates=True)
max_date = schema.select(pl.col("t_dat").max()).collect().item()
test_start = max_date - timedelta(days=6); valid_start = test_start - timedelta(days=7); rerank_train_start = valid_start - timedelta(days=7)
print({"max_date": str(max_date), "rerank_train_start": str(rerank_train_start), "valid_start": str(valid_start), "test_start": str(test_start)})

# Customer eligibility and strata use only train-period purchases. Hash order is
# deterministic and avoids selecting users because of future labels.
history = (schema.filter(pl.col("t_dat") < pl.lit(rerank_train_start))
    .group_by("customer_id").agg(pl.len().alias("n_train"))
    .filter(pl.col("n_train") >= cfg["min_train_events"])
    .with_columns((pl.col("n_train").log10().floor().clip(0, 4).cast(pl.Int8)).alias("activity_bin"))
    .with_columns(pl.struct(["customer_id"]).hash(seed=SEED).alias("sample_hash")))
counts = history.group_by("activity_bin").len().collect().sort("activity_bin")
total = counts["len"].sum()
quotas = {row["activity_bin"]: max(1, round(cfg["customers"] * row["len"] / total)) for row in counts.to_dicts()}
selected = pl.concat([history.filter(pl.col("activity_bin") == b).sort("sample_hash").head(q) for b, q in quotas.items()]).select("customer_id")
selected_path = WORK / "selected_customers.parquet"; selected.collect().write_parquet(selected_path)
print("selected users:", sum(quotas.values()), "bins:", quotas)

In [ ]:
# Keep all future interactions but cap only old training history per selected user.
events = (schema.join(pl.scan_parquet(selected_path), on="customer_id", how="inner")
    .with_columns(pl.col("t_dat").cast(pl.Date), pl.col("article_id").cast(pl.String))
    .with_columns(pl.when(pl.col("t_dat") < pl.lit(rerank_train_start)).then(pl.lit("train"))
        .when(pl.col("t_dat") < pl.lit(valid_start)).then(pl.lit("rerank_train"))
        .when(pl.col("t_dat") < pl.lit(test_start)).then(pl.lit("valid")).otherwise(pl.lit("test")).alias("split"))
    .with_columns(pl.when(pl.col("split") == "train").then(pl.lit(0)).otherwise(pl.lit(1)).alias("future"))
    .with_columns(pl.col("t_dat").rank("ordinal", descending=True).over("customer_id", "split").alias("recent_rank"))
    .filter((pl.col("split") != "train") | (pl.col("recent_rank") <= cfg["max_events_per_customer"]))
    .select("customer_id", "article_id", "t_dat", "price", "sales_channel_id", "split")
    .sort(["customer_id", "t_dat", "article_id"]))
for split in ("train", "rerank_train", "valid", "test"):
    path = WORK / f"{split}.parquet"
    events.filter(pl.col("split") == split).collect(streaming=True).write_parquet(path, compression="zstd")

# Product metadata has no interaction labels, so it is safe to retain all rows.
items = (pl.scan_csv(articles_csv, schema_overrides={"article_id": pl.String})
    .select("article_id", "prod_name", "product_type_name", "product_group_name", "graphical_appearance_name", "colour_group_name", "department_name", "section_name", "detail_desc")
    .rename({"article_id": "item_id"}).collect(streaming=True))
items.write_parquet(WORK / "items.parquet", compression="zstd")
counts = {s: pl.scan_parquet(WORK / f"{s}.parquet").select(pl.len().alias("events"), pl.col("customer_id").n_unique().alias("users"), pl.col("article_id").n_unique().alias("items")).collect().to_dicts()[0] for s in ("train", "rerank_train", "valid", "test")}
manifest = {"dataset": cfg["dataset"], "version": cfg["version"], "profile": cfg["profile"], "seed": SEED, "windows": {"retrieval_train_before": str(rerank_train_start), "reranker_train": [str(rerank_train_start), str(valid_start - timedelta(days=1))], "valid": [str(valid_start), str(test_start - timedelta(days=1))], "test": [str(test_start), str(max_date)]}, "sampling": {"customers": cfg["customers"], "min_train_events": cfg["min_train_events"], "max_old_train_events": cfg["max_events_per_customer"], "selection_uses": "before reranker-train window only"}, "counts": counts}
(WORK / "dataset_manifest.json").write_text(json.dumps(manifest, indent=2)); display(manifest)

Validation and test are global seven-day windows. Users with no target purchase in a window are retained in the events but excluded by the metric for that window.